In [1]:
"""
Week 2 - Data Cleaning and Preprocessing
Logistics Data Analysis Internship

"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler


df = pd.read_csv('ecommerce_logistics_delivery.csv')

print("Original shape:", df.shape)



print("\nMissing values per column:")
print(df.isnull().sum()[df.isnull().sum() > 0])

print("\nDuplicate Order_IDs:", df.duplicated(subset='Order_ID').sum())



num_cols = df.select_dtypes(include=[np.number]).columns
for col in num_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].median())

cat_cols = df.select_dtypes(include=['object']).columns
for col in cat_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].mode()[0])

print("\nMissing values after filling:", df.isnull().sum().sum())



before = len(df)
df = df.drop_duplicates(subset='Order_ID', keep='first')
print(f"Removed {before - len(df)} duplicate rows")



for col in df.select_dtypes(include=['object']).columns:
    df[col] = df[col].astype(str).str.strip().str.title()



cols_to_cap = ['Delivery_Delay_Hours', 'Shipping_Cost',
               'Distance_KM', 'Package_Weight_KG',
               'Warehouse_Processing_Hours']

for col in cols_to_cap:
    if col in df.columns:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR

        outliers = ((df[col] < lower) | (df[col] > upper)).sum()
        if outliers > 0:
            df[col] = df[col].clip(lower, upper)
            print(f"{col}: capped {outliers} outliers")



scaler = MinMaxScaler()
scale_cols = ['Delivery_Delay_Hours', 'Shipping_Cost',
              'Distance_KM', 'Package_Weight_KG']

for col in scale_cols:
    if col in df.columns:
        df[col + '_scaled'] = scaler.fit_transform(df[[col]])



df['Order_Date'] = pd.to_datetime(df['Order_Date'], errors='coerce')
df['day_of_week'] = df['Order_Date'].dt.day_name()
df['month'] = df['Order_Date'].dt.month
df['is_weekend'] = df['Order_Date'].dt.dayofweek.isin([5, 6]).astype(int)

df['cost_per_km'] = df['Shipping_Cost'] / df['Distance_KM']
df['cost_per_km'] = df['cost_per_km'].replace([np.inf, -np.inf], np.nan)
df['cost_per_km'] = df['cost_per_km'].fillna(df['cost_per_km'].median())


print("\nFinal shape:", df.shape)
print("New columns added:", ['day_of_week', 'month', 'is_weekend', 'cost_per_km'])

df.to_csv('logistics_data_cleaned.csv', index=False)


Original shape: (15000, 36)

Missing values per column:
Series([], dtype: int64)

Duplicate Order_IDs: 0

Missing values after filling: 0
Removed 0 duplicate rows


C:\Users\himan\AppData\Local\Temp\ipykernel_33492\394152870.py:32: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include=['object']).columns
C:\Users\himan\AppData\Local\Temp\ipykernel_33492\394152870.py:47: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide

Delivery_Delay_Hours: capped 117 outliers
Shipping_Cost: capped 342 outliers
Distance_KM: capped 352 outliers
Package_Weight_KG: capped 743 outliers
Warehouse_Processing_Hours: capped 476 outliers

Final shape: (15000, 44)
New columns added: ['day_of_week', 'month', 'is_weekend', 'cost_per_km']
